# Communication-Computation Efficient Gradient Coding

This notebook implements the toy example from the paper *Communication-Computation Efficient Gradient Coding* by Min Ye and Emmanuel Abbe.

It demonstrates the scheme where $n=5$ workers compute gradients for $k=5$ data subsets. By assigning $d=3$ subsets per worker, the system leverages a tradeoff to simultaneously tolerate $s=1$ straggler and reduce the communication dimension by a factor of $m=2$.

In [6]:
# Cell 2: Parameters & Imports
import numpy as np

# Parameters from the paper's toy example
n = 5  # Number of workers
k = 5  # Number of data subsets
d = 3  # Data subsets assigned to each worker
s = 1  # Number of stragglers tolerated
m = 2  # Communication reduction factor
l = 2  # Dimension of gradient vectors

# Verify the fundamental tradeoff condition: d >= s + m
assert d >= s + m, "Tradeoff condition not met."

# Evaluation roots (theta) associated with the n workers
# The paper suggests {0, +-1, +-1.5} for n=5 for numerical stability
theta = np.array([0.0, 1.0, -1.0, 1.5, -1.5])

print("Setup Complete. Parameters loaded.")

Setup Complete. Parameters loaded.


### Step 1: Recursive Polynomial and Matrix Construction
The core of the optimal scheme is constructing an $(mn) \times (n-s)$ matrix $B$ using recursive polynomials.

In [7]:
# Cell 3: Construct polynomials and Matrix B
P = []
for i in range(n):
    # Roots for p_i are theta_{i \oplus j} for j=1 to n-d
    # \oplus is 1-based modulo n, mapping to (i + j) % n for 0-based indexing
    roots_i = [theta[(i + j) % n] for j in range(1, n - d + 1)]
    p_i_1 = np.poly1d(roots_i, True) 
    
    P_i = [p_i_1]
    for u in range(2, m + 1):
        # Recursive polynomial generation
        prev_p = P_i[-1]
        deg_prev = n - d + u - 2
        target_deg = n - d - 1
        idx = deg_prev - target_deg
        
        coeff = prev_p.coeffs[idx] if (0 <= idx <= deg_prev) else 0.0
        p_i_u = np.poly1d([1, 0]) * prev_p - coeff * P_i[0]
        P_i.append(p_i_u)
    P.append(P_i)

# Construct the (m*n) x (n-s) matrix B
B = np.zeros((m * n, n - s))
for i in range(n):
    for u in range(m):
        poly = P[i][u]
        deg = poly.order
        for j in range(n - s):
            idx = deg - j
            if 0 <= idx <= deg:
                B[i * m + u, j] = poly.coeffs[idx]

# Precompute Explicit Encoding Weights per worker
Encoding_Weights = np.zeros((n, n, m)) # (worker i, dataset j, component u)
for i in range(n):
    A_i = np.array([theta[i]**p for p in range(n - s)])
    for j in range(n):
        for u in range(m):
            B_row = B[j * m + u, :]
            Encoding_Weights[i, j, u] = B_row @ A_i

print("Matrix B and Encoding Weights successfully constructed.")

Matrix B and Encoding Weights successfully constructed.


### Step 2: Data Simulation and Worker Transmissions
We generate random partial gradients and have each worker calculate its transmitted vector. Thanks to the polynomial roots mapping, a worker's encoding weights for unassigned datasets evaluate to exactly 0.

In [8]:
# Cell 4: Simulate partial gradients & Worker Encoding
np.random.seed(42)
G = np.random.rand(n, l) # n datasets, each of dimension l
true_sum = np.sum(G, axis=0)

print("--- Partial Gradients ---")
for j in range(n):
    print(f"Dataset {j}: {G[j]}")
print(f"\n>>> True Sum Gradient: {true_sum}\n")

# Workers compute their transmitted scalars (f_i)
# Because l/m = 1, each worker only transmits a single scalar.
F_workers = np.zeros(n)
for i in range(n):
    f_i = 0.0
    for j in range(n):
        # The polynomial logic ensures workers ONLY need assigned datasets
        if np.any(np.abs(Encoding_Weights[i, j]) > 1e-9):
            f_i += Encoding_Weights[i, j, 0] * G[j, 0] + Encoding_Weights[i, j, 1] * G[j, 1]
    F_workers[i] = f_i

print("--- Transmissions to Master ---")
for i in range(n):
    print(f"Worker {i} transmits scalar: {F_workers[i]:.4f}")

--- Partial Gradients ---
Dataset 0: [0.37454012 0.95071431]
Dataset 1: [0.73199394 0.59865848]
Dataset 2: [0.15601864 0.15599452]
Dataset 3: [0.05808361 0.86617615]
Dataset 4: [0.60111501 0.70807258]

>>> True Sum Gradient: [1.92175133 3.27961603]

--- Transmissions to Master ---
Worker 0 transmits scalar: -2.2726
Worker 1 transmits scalar: -2.9575
Worker 2 transmits scalar: 2.2559
Worker 3 transmits scalar: 4.2906
Worker 4 transmits scalar: -0.1879


### Step 3: Master Node Decoding
The Master waits for any $n-s = 4$ workers. It drops the straggler, sets up the linear system using a Vandermonde matrix, and isolates the sum gradient.

In [12]:
# Cell 5: Master node mitigates the straggler and recovers the sum
straggler_idx = 1  # Simulate Worker 3 failing/lagging
print(f"Worker {straggler_idx} is a straggler! Master ignores it and decodes using the remaining 5.\n")

survivors = [i for i in range(n) if i != straggler_idx]
F_surv = F_workers[survivors]

# Reconstruct the Vandermonde matrix for surviving workers
A_surv = np.array([[theta[i]**j for j in range(n - s)] for i in survivors])

# Recover the C vector by solving the linear system: A_surv @ C = F_surv
C_rec = np.linalg.solve(A_surv, F_surv)

# The sum gradient components correspond to the last m entries of C
# Specifically, indices (n - d) and (n - d + 1) 
recovered_sum = np.array([C_rec[n - d], C_rec[n - d + 1]])

print("--- Decoding Results ---")
print(f"Recovered Sum Gradient: {recovered_sum}")
print(f"Difference from True Sum: {np.linalg.norm(true_sum - recovered_sum):.2e}")
if np.linalg.norm(true_sum - recovered_sum) < 1e-9:
    print("\nSUCCESS: The coded sum matches the true sum precisely!")
else:
    print("\nFAILURE: Mismatch detected.")

Worker 1 is a straggler! Master ignores it and decodes using the remaining 5.

--- Decoding Results ---
Recovered Sum Gradient: [1.92175133 3.27961603]
Difference from True Sum: 9.16e-16

SUCCESS: The coded sum matches the true sum precisely!
